# 19. Run or resume the paired generalization study

Run notebook 18 in PREPARE mode first. This notebook reads its saved configuration.
Start with one work unit per session. Reopening it resumes verified unfinished work and
skips completed jobs. It trains source histories independently for every order and seed.

Learning saves after each epoch. Unlearning saves at the configured step interval.
A runtime loss can discard work since the last verified R2 boundary. An interrupted
learning epoch restarts from its preceding boundary with the saved Adam, protection
snapshot, random state, and data-order generator. It does not start with a fresh optimizer.

The time budget is a soft limit: a boundary and its uploads may finish after the limit.
Use a generous reserve. A GPU/software mismatch stops resume rather than silently changing
the experiment. Resume support does not establish repeatability across different GPUs.

## 1. Load the fixed code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading this fixed code revision.")
if IN_COLAB:
    REPO = Path("/content/cark-paired-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    implementation = ["uncle", "pyproject.toml", "requirements.txt"]
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", *implementation], text=True).strip()
    # A later documentation-only commit may contain the same fixed implementation.
    differences = subprocess.check_output(["git", "-C", str(REPO), "diff", CODE_VERSION, actual, "--", *implementation], text=True).strip()
    if dirty or differences:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open in Colab. Local training edits are preserved.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies before running this notebook.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))
from uncle.storage import R2Store
from uncle.paired_study import StudyStore, make_plan, jobs, inspect_queue, run_queue, review
import torch
print("Fixed implementation revision:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)


## 2. Set the session budget

In [ ]:
STUDY_ID = "paired_generalization_v1"
BUCKET = "uncle-experiments"
RUN_TRAINING = False  # Set True to execute this bounded GPU session.
MAX_JOBS = 1
MAX_SESSION_MINUTES = 120
SAVE_RESERVE_MINUTES = 15
CLEAN_COMPLETED_RESUME_SLOTS = True
RELEASE_GPU_WHEN_DONE = True
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
if not RUN_TRAINING:
    raise RuntimeError("Training is off. Set RUN_TRAINING = True when ready to run this session.")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime for this notebook.")
# These flags are recorded and must match on every resumed session.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print("GPU:", torch.cuda.get_device_name(0))
print("Session limit:", MAX_SESSION_MINUTES, "minutes; reserve:", SAVE_RESERVE_MINUTES)


## 3. Connect and recover the queue

In [ ]:
try:
    def secret(name):
        value = os.environ.get(name)
        if not value and IN_COLAB:
            from google.colab import userdata
            value = userdata.get(name)
        if not value:
            raise RuntimeError("Missing secret: " + name)
        return value
    store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
    store.progress = True
    CACHE = REPO / "outputs/paired_study" / STUDY_ID
    CACHE.mkdir(parents=True, exist_ok=True)
    print("Connected to R2 bucket:", BUCKET)
    # Transport-only fix: preserve the saved training revision and resume contract.
    from boto3.s3.transfer import TransferConfig
    from boto3.exceptions import S3UploadFailedError
    from botocore.exceptions import ClientError
    import time
    transfer_config = TransferConfig(multipart_threshold=64 * 1024 * 1024,
        multipart_chunksize=64 * 1024 * 1024, max_concurrency=1,
        use_threads=False, preferred_transfer_client="classic")
    original_upload_file = store.client.upload_file
    def upload_with_multipart_retry(*args, **kwargs):
        kwargs["Config"] = transfer_config
        for attempt in range(1, 4):
            try:
                return original_upload_file(*args, **kwargs)
            except (S3UploadFailedError, ClientError) as error:
                error_code = getattr(error, "response", {}).get("Error", {}).get("Code")
                invalid_part = error_code == "InvalidPart" or (
                    isinstance(error, S3UploadFailedError) and "(InvalidPart)" in str(error))
                if not invalid_part or attempt == 3:
                    raise
                print("R2 InvalidPart: restarting the full upload, attempt", attempt + 1,
                    "of 3. Saved progress has not advanced.", flush=True)
                callback_owner = getattr(kwargs.get("Callback"), "__self__", None)
                if callback_owner is not None and hasattr(callback_owner, "reset"):
                    callback_owner.reset()
                time.sleep(2 * attempt)
    store.client.upload_file = upload_with_multipart_retry
    print("R2 uploads: sequential 64 MiB parts; at most three InvalidPart attempts.")
except BaseException:
    import traceback
    traceback.print_exc(file=sys.stdout)
    sys.stdout.flush()
    if RELEASE_GPU_WHEN_DONE and IN_COLAB:
        from google.colab import runtime
        print("Setup failed before training. Releasing the GPU runtime.")
        runtime.unassign()
    raise


In [ ]:
try:
    PREFIX = "uncle/paired_generalization/" + STUDY_ID
    path = store.download(PREFIX + "/study.json", CACHE / "cache")
    plan = json.loads(path.read_text())
    if plan["code_version"] != CODE_VERSION:
        raise ValueError("This study requires a different fixed code revision. Use its recorded notebook revision.")
    from uncle.learning_diagnostic import file_hash
    if file_hash(REPO / "uncle/task_partition.json") != plan["partition_sha256"]:
        raise ValueError("Class partition differs from the saved study.")
    storage = StudyStore(store, plan, CACHE, cleanup_completed=CLEAN_COMPLETED_RESUME_SLOTS)
    rows = inspect_queue(plan, storage, progress=True)
    from collections import Counter
    print("Queue:", dict(Counter(row["status"] for row in rows)))
    for row in rows:
        print(row["status"].ljust(10), row["id"], row["active"] or "")
except BaseException:
    import traceback
    traceback.print_exc(file=sys.stdout)
    sys.stdout.flush()
    if RELEASE_GPU_WHEN_DONE and IN_COLAB:
        from google.colab import runtime
        print("Setup failed before training. Releasing the GPU runtime.")
        runtime.unassign()
    raise


## 4. Run the next work units

Tiny ImageNet is downloaded to local runtime storage only when needed, with a download
progress bar. It is reused across the session. Evaluation, learning, unlearning, R2 transfers,
and the study queue also have progress bars.

Every progress pointer is published after its checkpoint and report are verified.
Source models remain available for every branch. By default, branch final models are
deleted after the completed report and model verification receipt are durably saved.
Immediate post-unlearning models are not retained by default. All reports, settings,
hashes, and figures remain. Unfinished jobs retain resumable model and optimizer state.
Cleanup touches only this study's completed-job model files and temporary slots.
It never deletes historical experiment files.

In [ ]:
from uncle.streams import build_tasks
task_cache = {}
def tasks_factory(job):
    wanted = tuple(sorted(set(plan["orders"][job["order"]]) |
        ({job["new_task"]} if job["source"] else set())))
    if wanted not in task_cache:
        # Dataset preparation occurs inside the timed session, not before its budget.
        task_cache[wanted] = build_tasks(root=DATA_ROOT,
            partition_path=REPO / "uncle/task_partition.json", partition_seed=42,
            include=wanted, download=True)
    return task_cache[wanted]

session = None
try:
    session = run_queue(plan, storage, tasks_factory, max_jobs=MAX_JOBS,
        max_minutes=MAX_SESSION_MINUTES, reserve_minutes=SAVE_RESERVE_MINUTES,
        device="cuda", progress=True)
    print(json.dumps(session, indent=2))
    print("Session progress saved. Reopen this notebook with the same STUDY_ID to continue.")
except BaseException as error:
    import traceback
    traceback.print_exc(file=sys.stdout)
    sys.stdout.flush()
    print("Session interrupted:", type(error).__name__)
    print("Verified earlier R2 boundaries remain available. Do not change the study settings to resume.")
    raise
finally:
    task_cache.clear()
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    if RELEASE_GPU_WHEN_DONE and IN_COLAB:
        from google.colab import runtime
        print("Releasing the Colab runtime.")
        runtime.unassign()


## 5. Review on CPU

Open notebook 20 to inspect completed pairs, interruption records, averages, and figures.
It downloads reports rather than models. Do not occupy a GPU for analysis.